In [1]:
import pandas as pd
import numpy as np
import os

# Project path
project_path = r"C:\Users\HP\OneDrive\Desktop\AI Traffic Accident Risk Prediction"

# Dataset path
data_path = os.path.join(
    project_path,
    "data",
    "UK_Accident.csv"
)

# Load dataset
df = pd.read_csv(data_path)

print("=" * 70)
print("PRE-ACCIDENT MODEL - DATA LOADING")
print("=" * 70)

print("Dataset shape:", df.shape)

# Remove unnecessary columns
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

if "Accident_Index" in df.columns:
    df.drop(columns=["Accident_Index"], inplace=True)

# Remove rows where target is missing
df = df.dropna(subset=["Accident_Severity"])

print("\nDataset shape after cleaning:", df.shape)

print("\nTarget distribution:")
print(df["Accident_Severity"].value_counts())

print("\nData loaded and cleaned successfully ✅")

PRE-ACCIDENT MODEL - DATA LOADING
Dataset shape: (111, 33)

Dataset shape after cleaning: (111, 31)

Target distribution:
Accident_Severity
3    102
2      9
Name: count, dtype: int64

Data loaded and cleaned successfully ✅


In [2]:
# Features available for pre-accident prediction

features = [
    "Police_Force",
    "Speed_limit",
    "Light_Conditions",
    "Weather_Conditions",
    "Road_Surface_Conditions",
    "Special_Conditions_at_Site",
    "Carriageway_Hazards",
    "Urban_or_Rural_Area",
    "Year"
]

target = "Accident_Severity"

# Create input features and target
X = df[features].copy()
y = df[target].copy()

print("=" * 70)
print("PRE-ACCIDENT FEATURES")
print("=" * 70)

print("\nInput features:")
for feature in features:
    print("-", feature)

print("\nTarget:")
print("-", target)

print("\nFeature data shape:", X.shape)
print("Target shape:", y.shape)

print("\nFeatures selected successfully ✅")


PRE-ACCIDENT FEATURES

Input features:
- Police_Force
- Speed_limit
- Light_Conditions
- Weather_Conditions
- Road_Surface_Conditions
- Special_Conditions_at_Site
- Carriageway_Hazards
- Urban_or_Rural_Area
- Year

Target:
- Accident_Severity

Feature data shape: (111, 9)
Target shape: (111,)

Features selected successfully ✅


In [3]:
# Check missing values and data types

print("=" * 70)
print("DATA PREPARATION")
print("=" * 70)

print("\nMissing values:")
print(X.isnull().sum())

print("\nData types:")
print(X.dtypes)

print("\nTarget values:")
print(y.value_counts())

print("\nData preparation check completed ✅")

DATA PREPARATION

Missing values:
Police_Force                    0
Speed_limit                     0
Light_Conditions                0
Weather_Conditions              0
Road_Surface_Conditions         0
Special_Conditions_at_Site    108
Carriageway_Hazards           110
Urban_or_Rural_Area             0
Year                            0
dtype: int64

Data types:
Police_Force                  int64
Speed_limit                   int64
Light_Conditions                str
Weather_Conditions              str
Road_Surface_Conditions         str
Special_Conditions_at_Site      str
Carriageway_Hazards             str
Urban_or_Rural_Area           int64
Year                          int64
dtype: object

Target values:
Accident_Severity
3    102
2      9
Name: count, dtype: int64

Data preparation check completed ✅


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Identify numerical and categorical features
numeric_features = [
    "Police_Force",
    "Speed_limit",
    "Year"
]

categorical_features = [
    "Light_Conditions",
    "Weather_Conditions",
    "Road_Surface_Conditions",
    "Special_Conditions_at_Site",
    "Carriageway_Hazards",
    "Urban_or_Rural_Area"
]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Fit preprocessing on training data only
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data
X_test_processed = preprocessor.transform(X_test)

print("=" * 70)
print("DATA SPLIT AND PREPROCESSING")
print("=" * 70)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

print("\nOriginal training shape :", X_train.shape)
print("Processed training shape:", X_train_processed.shape)

print("\nOriginal testing shape :", X_test.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nPreprocessing completed successfully ✅")

DATA SPLIT AND PREPROCESSING
Training samples: 88
Testing samples : 23

Original training shape : (88, 9)
Processed training shape: (88, 22)

Original testing shape : (23, 9)
Processed testing shape: (23, 22)

Preprocessing completed successfully ✅


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

print("=" * 70)
print("PRE-ACCIDENT MODEL TRAINING")
print("=" * 70)

# Define models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )
}

results = []

# Train and evaluate each model
for name, model in models.items():

    print(f"\nTraining {name}...")

    model.fit(
        X_train_processed,
        y_train
    )

    y_pred = model.predict(
        X_test_processed
    )

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1
    })

# Create results table
results_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

display(results_df)

# Select the best model based on F1-score
best_model_name = results_df.loc[
    results_df["F1-Score"].idxmax(),
    "Model"
]

best_model = models[best_model_name]

print("\nBest Model:", best_model_name)

# Save model
model_path = os.path.join(
    project_path,
    "models",
    "pre_accident_model.pkl"
)

preprocessor_path = os.path.join(
    project_path,
    "models",
    "pre_accident_preprocessor.pkl"
)

feature_path = os.path.join(
    project_path,
    "models",
    "pre_accident_features.pkl"
)

joblib.dump(
    best_model,
    model_path
)

joblib.dump(
    preprocessor,
    preprocessor_path
)

joblib.dump(
    features,
    feature_path
)

print("\nFiles saved successfully:")
print("Model        :", model_path)
print("Preprocessor :", preprocessor_path)
print("Features     :", feature_path)

print("\nPre-accident model training completed successfully ✅")

PRE-ACCIDENT MODEL TRAINING

Training Logistic Regression...

Training Decision Tree...

Training Random Forest...

MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.913043,0.833648,0.913043,0.871542
1,Decision Tree,0.913043,0.833648,0.913043,0.871542
2,Random Forest,0.913043,0.833648,0.913043,0.871542



Best Model: Logistic Regression

Files saved successfully:
Model        : C:\Users\HP\OneDrive\Desktop\AI Traffic Accident Risk Prediction\models\pre_accident_model.pkl
Preprocessor : C:\Users\HP\OneDrive\Desktop\AI Traffic Accident Risk Prediction\models\pre_accident_preprocessor.pkl
Features     : C:\Users\HP\OneDrive\Desktop\AI Traffic Accident Risk Prediction\models\pre_accident_features.pkl

Pre-accident model training completed successfully ✅
